## Initialization

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

import os

import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
import datetime
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import seaborn as sns
from pandas.plotting import register_matplotlib_converters

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from keras.models import Sequential, load_model
from keras.layers import Input, Dense, LSTM, Dropout, Conv1D, GlobalAveragePooling1D, Reshape
import keras_tuner as kt
from keras import backend as K
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.regularizers import l2

import yfinance as yf

from IPython.display import display, clear_output
from IPython import get_ipython


In [ ]:
# Set your ticker and folder
ticker = "DOGE-USD"
maindir = '/Users/ajfabbri/Library/CloudStorage/OneDrive-TheColoradoCollege/Thesis/Lab Oct 2024'
folder = os.path.join(maindir, "datacache")  # <-- Replace with your folder path
file_path = os.path.join(folder, f"{ticker}.xlsx")

# Calculate the date range: last 20 years until today
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=365 * 20)).strftime('%Y-%m-%d')

# Create the yfinance Ticker object
tk = yf.Ticker(ticker)

def add_dividends_splits(data_df):
    # If data_df has multi-level columns, flatten them
    if isinstance(data_df.columns, pd.MultiIndex):
        data_df.columns = data_df.columns.get_level_values(0)
    
    # Reset index to bring the Date into a column
    data_df = data_df.reset_index()
    
    # Get dividends and remove timezone information
    dividends = tk.dividends.reset_index()
    dividends['Date'] = dividends['Date'].dt.tz_localize(None)
    
    # Get splits and remove timezone information
    splits = tk.splits.reset_index()
    splits['Date'] = splits['Date'].dt.tz_localize(None)
    
    # Merge the dividend and splits data into the main dataframe
    merged_df = data_df.merge(dividends, on="Date", how="left") \
                         .merge(splits, on="Date", how="left")
    
    # Fill missing values for dividends and stock splits
    merged_df['Dividends'] = merged_df['Dividends'].fillna(0)
    merged_df['Stock Splits'] = merged_df['Stock Splits'].fillna(1)
    
    # Set Date as index and calculate daily returns
    merged_df.set_index("Date", inplace=True)
    merged_df['Return'] = merged_df['Close'].pct_change()
    return merged_df

# Check if the Excel file exists
if os.path.exists(file_path):
    # Load existing data (assumes Date is the index)
    existing_df = pd.read_excel(file_path, index_col="Date", parse_dates=True)
    
    # Delete any rows before start_date
    existing_df = existing_df[existing_df.index >= pd.to_datetime(start_date)]
    
    # Determine the most recent date in the existing data
    if not existing_df.empty:
        last_date = existing_df.index.max().date()
        # Download new data starting from the day after the last date
        new_start_date = (last_date + timedelta(days=1)).strftime('%Y-%m-%d')
    else:
        new_start_date = start_date

    # Only download new data if new_start_date is not after end_date
    if new_start_date <= end_date:
        new_data = yf.download(ticker, start=new_start_date, end=end_date)
        if not new_data.empty:
            new_data = add_dividends_splits(new_data)
            # Combine the existing data with the new data
            df = pd.concat([existing_df, new_data])
        else:
            df = existing_df.copy()
    else:
        df = existing_df.copy()
else:
    # File doesn't exist; download the full 20-year history
    full_data = yf.download(ticker, start=start_date, end=end_date)
    df = add_dividends_splits(full_data)

# Save (or overwrite) the Excel file with the updated dataframe
df.to_excel(file_path)
print(f"Data for {ticker} updated and saved to {file_path}")

In [ ]:
# Read and process FF data
ff_data = pd.read_csv(os.path.join(maindir, "data", "FF_Factors.csv"))
# Adjust the format if necessary; for example, if dates are in 'YYYY-MM-DD', remove the format argument:
# Convert 'Date' column to datetime (assuming YYYYMMDD format)
ff_data['Date'] = pd.to_datetime(ff_data['Date'], format='%Y%m%d')

# Filter FF data to match your date range from the prior block:
start_date_dt = pd.to_datetime(start_date)
end_date_dt = pd.to_datetime(end_date)
ff_data = ff_data[(ff_data['Date'] >= start_date_dt) & (ff_data['Date'] <= end_date_dt)]

# Inspect FF data
print("FF data head:")
print(ff_data.head())
print("FF data date range:", ff_data['Date'].min(), ff_data['Date'].max())

# Merge using merge_asof (if the dates don’t match exactly)
df = df.reset_index()
df = pd.merge_asof(df.sort_values('Date'), ff_data.sort_values('Date'), on='Date', direction='backward')
df = df.set_index('Date')

# Forward-fill the FF factor columns
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
df[factor_columns] = df[factor_columns].ffill()

print("Merged DataFrame head:")
print(df.head())

In [ ]:
print(df['Dividends'].describe())
print(df['Stock Splits'].describe())

In [ ]:
print(df.head())
plot_acf(df['Close'], lags=100)
plt.show()

In [ ]:
register_matplotlib_converters()
sns.set(font_scale=1.5, style="whitegrid")

In [ ]:
null_columns=df.columns[df.isnull().any()]
df[null_columns].isnull().sum()

df.dropna(inplace=True)

## Data

In [ ]:
df.plot(subplots = True, figsize = (10,12))
plt.title(ticker + ' Stock Attributes')
plt.show()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df['Return'].plot(legend=True, linestyle='--', marker='o')

In [ ]:
plt.figure(figsize=(16,6))
plt.title('Close Price History')
plt.plot(df['Close'])
plt.xlabel('Date', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)
plt.show()

## Scaling and data splitting

In [ ]:
cols = ["Open","High","Low","Volume","Dividends","Stock Splits"]
cols_ff = ["Open","High","Low","Volume","Dividends","Stock Splits","Mkt-RF","SMB","HML","RMW","CMA","RF"]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[cols_ff])

scaler_y = MinMaxScaler()
scaled_data_y = scaler_y.fit_transform(df[["Close"]])

In [ ]:
def create_sliding_windows(X, y, timesteps, n_future):
    X_sliding = []
    y_sliding = []
    for i in range(len(X) - timesteps - n_future + 1):
        X_sliding.append(X[i:i + timesteps])
        y_sliding.append(y[i + timesteps:i + timesteps + n_future])
    return np.array(X_sliding), np.array(y_sliding)

# First split to separate out the test set
X_temp0, X_test0, y_temp0, y_test0 = train_test_split(
    df[cols_ff], df[["Close"]], test_size=0.1, random_state=42, shuffle=False
)
# Second split to separate out the validation set from the remaining data
X_train0, X_val0, y_train0, y_val0 = train_test_split(
    X_temp0, y_temp0, test_size=0.1, random_state=42, shuffle=False
)

# First split to separate out the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    scaled_data, scaled_data_y, test_size=0.1, random_state=42, shuffle=False
)
# Second split to separate out the validation set from the remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1, random_state=42, shuffle=False
)

# Convert to NumPy arrays
X_train, X_test, X_val = np.array(X_train), np.array(X_test), np.array(X_val)
y_train, y_test, y_val = np.array(y_train).flatten(), np.array(y_test).flatten(), np.array(y_val).flatten()

# length of input sequence 
timesteps = 15
# length of output sequence
n_future = 5

X_train_sliding, y_train_sliding = create_sliding_windows(X_train, y_train, timesteps, n_future)
X_val_sliding, y_val_sliding = create_sliding_windows(X_val, y_val, timesteps, n_future)
X_test_sliding, y_test_sliding = create_sliding_windows(X_test, y_test, timesteps, n_future)

# print(f"X_train_sliding shape: {X_train_sliding.shape}")  # (num_samples, timesteps, num_features)
# print(f"y_train_sliding shape: {y_train_sliding.shape}")  # (num_samples,)
# print(f"X_test_sliding shape: {X_test_sliding.shape}")    # (num_samples, timesteps, num_features)
# print(f"y_test_sliding shape: {y_test_sliding.shape}")    # (num_samples,)

In [ ]:
def create_sliding_windows(X, y, timesteps, n_future):
    X_sliding = []
    y_sliding = []
    for i in range(len(X) - timesteps - n_future + 1):
        X_sliding.append(X[i:i + timesteps])
        y_sliding.append(y[i + timesteps:i + timesteps + n_future])
    return np.array(X_sliding), np.array(y_sliding)

# First split to separate out the test set
X_temp0, X_test0, y_temp0, y_test0 = train_test_split(
    df[cols], df[["Close"]], test_size=0.1, random_state=42, shuffle=False
)
# Second split to separate out the validation set from the remaining data
X_train0, X_val0, y_train0, y_val0 = train_test_split(
    X_temp0, y_temp0, test_size=0.1, random_state=42, shuffle=False
)

# First split to separate out the test set
X_temp, X_test, y_temp, y_test = train_test_split(
    scaled_data, scaled_data_y, test_size=0.1, random_state=42, shuffle=False
)
# Second split to separate out the validation set from the remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1, random_state=42, shuffle=False
)

# Convert to NumPy arrays
X_train, X_test, X_val = np.array(X_train), np.array(X_test), np.array(X_val)
y_train, y_test, y_val = np.array(y_train).flatten(), np.array(y_test).flatten(), np.array(y_val).flatten()

# length of input sequence 
timesteps = 15
# length of output sequence
n_future = 5

X_train_sliding, y_train_sliding = create_sliding_windows(X_train, y_train, timesteps, n_future)
X_val_sliding, y_val_sliding = create_sliding_windows(X_val, y_val, timesteps, n_future)
X_test_sliding, y_test_sliding = create_sliding_windows(X_test, y_test, timesteps, n_future)

# print(f"X_train_sliding shape: {X_train_sliding.shape}")  # (num_samples, timesteps, num_features)
# print(f"y_train_sliding shape: {y_train_sliding.shape}")  # (num_samples,)
# print(f"X_test_sliding shape: {X_test_sliding.shape}")    # (num_samples, timesteps, num_features)
# print(f"y_test_sliding shape: {y_test_sliding.shape}")    # (num_samples,)

In [ ]:
cols = ["Open","High","Low","Volume","Dividends","Stock Splits"]

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[cols])

scaler_y = MinMaxScaler()
scaled_data_y = scaler_y.fit_transform(df[["Close"]])

## Building the model

In [ ]:
from keras.optimizers import Adam

def model_builder(hp):
    model = Sequential()
    model.add(Input(shape=(X_train_sliding.shape[1], X_train_sliding.shape[2])))
    
    model.add(LSTM(hp.Int('input_unit', min_value=32, max_value=512, step=32), return_sequences=True))
    for i in range(hp.Int('n_layers', 1, 4)):
        model.add(LSTM(hp.Int(f'lstm_{i}_units', min_value=32, max_value=512, step=32), return_sequences=True))
        
    model.add(LSTM(hp.Int('final_lstm_units', min_value=32, max_value=512, step=32)))
    model.add(Dropout(hp.Float('Dropout_rate', min_value=0.1, max_value=0.5, step=0.1)))
    model.add(Dense(hp.Int('dense_neurons', min_value=32, max_value=512, step=32),
        activation=hp.Choice('dense_activation', values=['relu', 'tanh'], default='relu')))
    model.add(Dense(n_future, activation='linear'))

    optimizer = Adam(learning_rate=0.001, clipvalue=1.0)
    model.compile(loss='mean_squared_error', optimizer=optimizer, metrics=['mse'])
    
    return model

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

tuner = kt.BayesianOptimization(model_builder, objective="mse", max_trials=3, executions_per_trial=1, directory="./LSTM/", overwrite=True)

In [ ]:
try:
    tuner.search(x=X_train_sliding, y=y_train_sliding, epochs = 30, batch_size = 32, validation_data=(X_val_sliding, y_val_sliding), callbacks=[early_stop])
except Exception as e:
    print(f"An error occurred during tuning: {e}")

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
model_path = f"./LSTM/{ticker}_noFF.keras"

best_model.save(model_path)
print(f"Best model saved at: {model_path}")

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
model_path = f"./LSTM/{ticker}.keras"

best_model.save(model_path)
print(f"Best model saved at: {model_path}")

In [ ]:
tuner.results_summary()

## Predictions and evaluation

In [ ]:
best_model = load_model(f'./LSTM/{ticker}_noFF.keras')
print(f'Loaded {ticker}_noFF.keras') 

In [ ]:
best_model = load_model(f'./LSTM/{ticker}.keras')
print(f'Loaded {ticker}.keras') 

In [ ]:
y_pred = best_model.predict(X_test_sliding)
y_pred.shape

In [ ]:
# Reshape y_pred if necessary
y_pred1 = scaler_y.inverse_transform(y_pred)
# Reshape y_test to a 2D array before inverse transforming
y_test1 = scaler_y.inverse_transform(y_test_sliding)

# Check that the lengths match
print(f"Length of y_pred: {len(y_pred1)}")
print(f"Length of y_test: {len(y_test1)}")


In [ ]:
days_future = 5

y_pred_n_step = y_pred1[:, days_future-1] #change for n days out (zero indexing)
y_test_n_step = y_test1[:, days_future-1] # change to index correctly, but keeping at zero should work.
print(f"Length of y_pred_nth_step: {len(y_pred_n_step)}")
print(f"Length of y_test_nth_step: {len(y_test_n_step)}")

# Create DataFrame with actual and predicted values
valid = pd.DataFrame({'Actual': y_test_n_step, 'Predicted': y_pred_n_step})

# Display the first few rows
print(valid.tail())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 6))
plt.title('Model')
plt.xlabel('Sample Index', fontsize=18)
plt.ylabel('Close Price USD ($)', fontsize=18)

# Plot training data
plt.plot(range(len(y_train0)), y_train0, label='Train')
validation_start_index = len(y_train0)+len(y_val0)
validation_indices = range(validation_start_index, validation_start_index + len(y_test0))
plt.plot(validation_indices, y_test0, label='Val Actual')

# Adjust indices for validation predictions to account for timesteps
prediction_start_index = validation_start_index + timesteps + days_future
prediction_indices = range(prediction_start_index, prediction_start_index + len(valid['Predicted']))
# Plot validation predictions with adjusted indices
plt.plot(prediction_indices, valid['Predicted'], label='Val Predicted')

# Add legend
plt.legend(loc='upper left')
# Show the plot
plt.show()

In [ ]:
# y_test = scaler_y.inverse_transform(y_test)

# Calculate evaluation metrics
mse = mean_squared_error(y_test_n_step, y_pred_n_step)
mae = mean_absolute_error(y_test_n_step, y_pred_n_step)
r2 = r2_score(y_test_n_step, y_pred_n_step)

# Print evaluation results
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

# Plotting predictions vs actual values
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test_n_step, label='Actual Values', color='b')
plt.plot(y_pred_n_step, label='Predicted Values', color='r')
plt.title('Actual vs Predicted Values')
plt.xlabel('Samples')
plt.ylabel('Close Price USD ($)')
plt.legend()
plt.show()

## Predicting the future??

In [ ]:
start_index = y_train_sliding.shape[0] + y_val_sliding.shape[0]
end_index = start_index + y_test_sliding.shape[0]

folder = os.path.join(maindir, "datacache")  # <-- Replace with your folder path
file_path = os.path.join(folder, f"{ticker}.xlsx")
df = pd.read_excel(file_path)
df = df.reset_index()

# Calculate daily returns and volatility
df['Returns'] = df['Close'].pct_change()
df['Volatility'] = df['Returns'].rolling(window=30).std()

# Instead of dropping rows, fill the NaN values (e.g., using backfill or a fixed value)
df['Volatility'] = df['Volatility'].bfill()

# Now, slice the dates exactly as before:
test_dates = df['Date'].iloc[start_index:end_index].reset_index(drop=True)

# This assertion should now pass because the number of rows matches your original data used for predictions
assert len(test_dates) == len(y_pred1), "Mismatch in lengths of test_dates and y_pred1"

# Generate buy/sell signals based on volatility thresholds
signals = []

# number of standard deviations to set the threshold
volatility_multiplier = 2

for i in range(len(y_pred1)):
    # Parameters
    offset = n_future+1  # Number of sequences ago for the current price
    
    # Ensure y_pred1 has enough sequences
    n_sequences = y_pred1.shape[0]
    if n_sequences < offset:
        raise ValueError(f"Not enough prediction data to compute current price with an offset of {offset} sequences.")

    date_now = test_dates[i]
    price_now = y_pred1[i -offset, -1]
    price_future = y_pred1[i, -1]
    delta_price = price_future - price_now
    pct_change = (delta_price / price_now) * 100

    # Get the volatility for the current date
    volatility_row = df[df['Date'] == date_now]
    if volatility_row.empty:
        continue  # Skip if date not found in df
    volatility = volatility_row['Volatility'].values[0] * 100  # Convert to percentage

    # Ensure volatility is not zero
    if volatility == 0:
        continue

    # Thresholds based on volatility
    buy_vol = volatility * 2
    sell_vol = -volatility_multiplier * volatility

    # Generate signals
    if pct_change >= buy_vol:
        signal_type = 'Buy'
    elif pct_change <= sell_vol:
        signal_type = 'Sell'
    else:
        continue  # No signal

    signals.append({
        'Date': date_now,
        'Signal': signal_type,
        'Predicted_Price_Now': price_now,
        'Predicted_Price_in_5_Days': price_future,
        'Predicted_Pct_Change': pct_change,
        'Volatility': volatility
    })

# Convert signals to a DataFrame
# signals_df = pd.DataFrame(signals)# Define the expected columns for the signals DataFrame.
signal_columns = ['Date', 'Signal', 'Predicted_Price_Now', 
                  'Predicted_Price_in_5_Days', 'Predicted_Pct_Change', 'Volatility']

# Create the signals DataFrame, even if 'signals' is empty.
signals_df = pd.DataFrame(signals, columns=signal_columns)


# Ensure 'Date' in df is of datetime type
df['Date'] = pd.to_datetime(df['Date'])
# Ensure 'Date' in signals_df is of datetime type
signals_df['Date'] = pd.to_datetime(signals_df['Date'])

# Merge the signals back to the original data for backtesting
df_signals = df.merge(signals_df[['Date', 'Signal']], on='Date', how='left')

df_filtered = df_signals.iloc[start_index:].reset_index(drop=True)

# Optionally, plot the signals on the price chart
plt.figure(figsize=(14, 7))
plt.plot(df_filtered['Date'], df_filtered['Close'], label='Close Price')
buy_signals = df_filtered[df_filtered['Signal'] == 'Buy']
sell_signals = df_filtered[df_filtered['Signal'] == 'Sell']
plt.scatter(buy_signals['Date'], buy_signals['Close'], marker='^', color='g', label='Buy Signal', s=100)
plt.scatter(sell_signals['Date'], sell_signals['Close'], marker='v', color='r', label='Sell Signal', s=100)
plt.title('Stock Price with Buy/Sell Signals')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
# Filter df_signals to include only the dates in df_filtered
df_filtered_signals = df_signals[df_signals['Date'].isin(df_filtered['Date'])].reset_index(drop=True)

# Initialize variables for the trading strategy
initial_cash = 1000.0
cash = initial_cash
holdings = 0.0  # Number of shares (positive for long, negative for short)
portfolio_values = []
dates = []
positions = []
last_buy_price = None  # To track the most recent buy price (including covering shorts)
last_sell_price = None  # To track the most recent sell price (including opening shorts)

# Adjustable parameters (you can adjust these)
buy_threshold = 0.95 # Buy if the price is below this factor of last sell price
sell_threshold = 1.05  # Sell if the price is above this factor of last buy price
stop_loss_threshold_short = 1.05

last_trade_date = None

for index, row in df_filtered_signals.iterrows():
    date = row['Date']
    price = row['Close']
    signal = row['Signal']

    # Implement stop-loss for short positions, above the 10 day check because we want the stop loss to trigger even if it's been less than 10 days.
    if holdings < 0 and price > last_sell_price * stop_loss_threshold_short:
        # Stop-loss triggered for short position
        cash += holdings * price  # holdings is negative
        print(f"{date.date()}: Stop-loss triggered - Covered short position of {abs(holdings):.4f} shares at ${price:.2f}")
        holdings = 0.0
        last_buy_price = price 
        last_trade_date = date

    # Check if enough time has passed since the last trade
    if last_trade_date is not None and (date - last_trade_date).days <= 10:
        continue  # Skip this iteration if less than 10 business days have passed

    if signal == 'Buy':
        # If holding a short position, cover it first
        if holdings < 0:
            if last_sell_price is not None and price < last_sell_price * buy_threshold:
                # Cover short position
                cash += holdings * price  # holdings is negative
                print(f"{date.date()}: Buy signal - Covered short position of {abs(holdings):.4f} shares at ${price:.2f}")
                holdings = 0.0
                last_buy_price = price
                last_trade_date = date
            else:
                # If price does not meet the threshold to cover, skip
                continue

        # Open a long position if currently not holding any shares
        if holdings == 0:
            # If there's a last sell price, enforce the threshold condition
            if last_sell_price is not None:
                if price < last_sell_price * buy_threshold:
                    holdings = cash / price
                    cash = 0.0
                    last_buy_price = price
                    last_trade_date = date
                    print(f"{date.date()}: Buy signal - Bought {holdings:.4f} shares at ${price:.2f}")
                else:
                    # Skip the buy because price is not below the threshold
                    continue
            else:
                # No last sell price exists; execute the buy immediately
                holdings = cash / price
                cash = 0.0
                last_buy_price = price
                last_trade_date = date
                print(f"{date.date()}: Buy signal - Bought {holdings:.4f} shares at ${price:.2f}")


    elif signal == 'Sell':
        # If holding a long position, close it first
        if holdings > 0:
            if last_buy_price is not None and price > last_buy_price * sell_threshold:
                cash = holdings * price
                print(f"{date.date()}: Sell signal - Sold holdings of {holdings:.4f} shares at ${price:.2f}, Cash: ${cash:.2f}")
                holdings = 0.0
                last_sell_price = price
                last_trade_date = date
            else:
                # If price does not meet the threshold to sell, skip
                continue

        # Open a short position
        if holdings == 0:
            if last_buy_price is not None and price > last_buy_price * sell_threshold:
                holdings = -(cash / price)
                cash += abs(holdings) * price
                last_sell_price = price
                last_trade_date = date
                print(f"{date.date()}: Sell signal - Opened short position of {abs(holdings):.4f} shares at ${price:.2f}, Portfolio Value: ${portfolio_value:.2f}")

    # Calculate portfolio value
    if holdings >= 0:
        portfolio_value = cash + holdings * price
    else:
        # For short positions, the value of the shorted shares is a liability
        portfolio_value = cash + holdings * price  # holdings is negative

    portfolio_values.append(portfolio_value)
    dates.append(date)
    positions.append({
        'Date': date,
        'Cash': cash,
        'Holdings': holdings,
        'Portfolio_Value': portfolio_value,
        'Price': price,
        'Signal': signal
    })

# Create DataFrame of portfolio values
portfolio_df = pd.DataFrame(positions)

# Calculate total return of the trading strategy
total_return = ((portfolio_df['Portfolio_Value'].iloc[-1] - initial_cash) / initial_cash) * 100
print(f"\nTotal Return of Trading Strategy: {total_return:.2f}%")

# Simulate buy-and-hold strategy
first_price = df_filtered['Close'].iloc[0]
holdings_bh = initial_cash / first_price
cash_bh = 0.0
portfolio_values_bh = []
for index, row in df_filtered.iterrows():
    date = row['Date']
    price = row['Close']
    portfolio_value_bh = holdings_bh * price + cash_bh
    portfolio_values_bh.append(portfolio_value_bh)

# Create DataFrame for buy-and-hold
bh_df = pd.DataFrame({
    'Date': df_filtered['Date'],
    'Buy_and_Hold': portfolio_values_bh
})

# Calculate total return of the buy-and-hold strategy
total_return_bh = ((bh_df['Buy_and_Hold'].iloc[-1] - initial_cash) / initial_cash) * 100
print(f"Total Return of Buy-and-Hold Strategy: {total_return_bh:.2f}%")

# Combine the two DataFrames for plotting
combined_df = portfolio_df[['Date', 'Portfolio_Value']].merge(bh_df, on='Date')

# Plot the portfolio values
plt.figure(figsize=(14, 7))
plt.plot(combined_df['Date'], combined_df['Portfolio_Value'], label='Trading Strategy')
plt.plot(combined_df['Date'], combined_df['Buy_and_Hold'], label='Buy and Hold')
plt.title('Portfolio Value Over Time (Filtered Dates)')
plt.xlabel('Date')
plt.ylabel('Portfolio Value ($)')
plt.legend()
plt.show()


In [ ]:
# save info
# 1 for true, 0 for false
ff_yn = 1
# 0=high info, 1=low info, 2=crypto
low_info = 1
# Variables to save
trial_data = {
    'Ticker': ticker,
    'MSE': mse,
    'MAE': mae,
    'R2': r2,
    'Total_Return': total_return,
    'Total_Return_BH': total_return_bh,
    'FF_YN': ff_yn,
    'Low_info': low_info,
    'Alpha': total_return-total_return_bh
}

# File path for Excel output
file_path = os.path.join(maindir, "trials_results.xlsx")

# Convert trial data to DataFrame
trial_df = pd.DataFrame([trial_data])

# Check if file exists
if os.path.exists(file_path):
    # If file exists, append new trial data
    existing_df = pd.read_excel(file_path)
    updated_df = pd.concat([existing_df, trial_df], ignore_index=True)
else:
    # If file doesn't exist, create a new one
    updated_df = trial_df

# Save the updated DataFrame to Excel
updated_df.to_excel(file_path, index=False)

print(f"Trial data appended and saved to {file_path}")

In [ ]:
# Ensure that the index of df is datetime
df.index = pd.to_datetime(df.index)

# Get the latest date from df
if 'Date' in df.columns:
    date_now = df['Date'].iloc[-1]
else:
    date_now = df.index[-1]  # Assuming the index contains dates

# Rest of your code
# Parameters
offset = n_future + 1  # Number of sequences ago for the current price

# Ensure y_pred1 has enough sequences
n_sequences = y_pred1.shape[0]
if n_sequences < offset:
    raise ValueError(f"Not enough prediction data to compute current price with an offset of {offset} sequences.")

# Note: test_dates, df, and other variables should already be defined as in your original code

signals = []
volatility_multiplier = 2

for i in range(len(y_pred1)):
    # We need to ensure we have enough history to get the current price based on the offset
    if i - offset < 0:
        # Skip until we have enough previous sequences to determine current_price
        continue

    # Compute current_price and future_forecast using indexing logic
    current_price = y_pred1[i - offset, -1]  # price from offset sequences ago
    future_forecast = y_pred1[i, -1]         # future price prediction for the current sequence


# Compute percentage change
delta = future_forecast - current_price
pct_change = (delta / current_price) * 100  # Percentage change based on current price

# Get the most recent 30 days of predictions (assume last column represents closing prices)
recent_predictions = y_pred1[-30:, -1]

# Calculate daily returns from the predictions
daily_returns = np.diff(recent_predictions) / recent_predictions[:-1]

# Calculate the rolling standard deviation (volatility)
volatility = np.std(daily_returns) * 100  # Convert to percentage
volatility_multiplier = 2

# Thresholds based on volatility
buy_vol = volatility * 2
sell_vol = -volatility_multiplier * volatility

# Print thresholds and predictions
print(f"Current Model Price (from sequence ending {offset-1} sequences ago): {current_price}")
print(f"Future Forecast (from latest sequence): {future_forecast}")
print(f"Percentage Change: {pct_change:.2f}%")
print(f"Buy Threshold (Volatility): {buy_vol:.2f}%")
print(f"Sell Threshold (Volatility): {sell_vol:.2f}%")

# Generate signals
if pct_change >= buy_vol:
    signal = 'Buy'
elif pct_change <= sell_vol:
    signal = 'Sell'
else:
    signal = 'None'  # No signal

print(f"Signal: {signal}")

# Use date_now for the Date in results
results = []
results.append({
    'Ticker': ticker,
    'Date': date_now,  # Use the latest date from df
    'Signal': signal,
    'Predicted ∆': pct_change,
    'Buy volatility': buy_vol,
    'Sell volatility': sell_vol
})

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

def save_to_excel(results_df, output_file):
    if os.path.exists(output_file):
        # Load the existing workbook
        wb = load_workbook(output_file)
        ws = wb.active  # Use the active sheet or specify the desired sheet

        # Convert DataFrame rows into lists and append to the worksheet without headers
        rows = dataframe_to_rows(results_df, index=False, header=False)
        for r in rows:
            ws.append(r)

        # Save the changes
        wb.save(output_file)
    else:
        # If the file doesn't exist, just create a new one using pandas, including headers
        results_df.to_excel(output_file, index=False)

    print(f"Signals saved to {output_file}")

output_file = './signals_output.xlsx'
save_to_excel(results_df, output_file)